# Notebook 50 – Modellvergleich, Evaluation & Business Insights
**Projekt:** Wirtschaftlichkeit von Airbnb-Listings in Spanien  
**Autorin:** Jasmin Müller (951624)  

---
### Struktur dieses Notebooks
1. Metriken aller drei Modelle zusammenführen
2. Vergleichstabelle & Vergleichsplot
3. Feature Importance (Modell III)
4. Residuenanalyse
5. Business Insights & Handlungsempfehlungen
6. Alle Outputs speichern (für LaTeX-Arbeit)

## 0 – Imports & Konfiguration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import pickle
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUTPUT_DIR = 'outputs/'
CACHE_DIR  = 'cache/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Einheitliches Plot-Styling ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':       150,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
COLORS = ['#4878CF', '#6ACC65', '#D65F5F']   # Blau, Grün, Rot → Modell I, II, III

print('Setup abgeschlossen ✓')

## 1 – Metriken aller drei Modelle

> **TODO:** Trage hier die Metriken von Annika (Modell I) und Mareike (Modell II) ein,
> sobald diese vorliegen. Nur die drei Zeilen im Dictionary anpassen – nichts anderes.
> Modell III ist bereits mit echten Werten befüllt.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TODO – Werte von Annika (Modell I) und Mareike (Modell II) hier eintragen
# Format: {'mae': float, 'rmse': float, 'r2': float}
# ══════════════════════════════════════════════════════════════════════════════

METRICS = {
    'Modell I\n(Lineare Regression)': {
        'mae':  999.99,    # ← TODO: Wert von Annika eintragen
        'rmse': 999.99,    # ← TODO: Wert von Annika eintragen
        'r2':   0.0000,    # ← TODO: Wert von Annika eintragen
        'color': COLORS[0],
        'notebook': 'NB10 – Annika'
    },
    'Modell II\n(Random Forest)': {
        'mae':  999.99,    # ← TODO: Wert von Mareike eintragen
        'rmse': 999.99,    # ← TODO: Wert von Mareike eintragen
        'r2':   0.0000,    # ← TODO: Wert von Mareike eintragen
        'color': COLORS[1],
        'notebook': 'NB20/30 – Mareike'
    },
    'Modell III\n(Gradient Boosting)': {
        'mae':  66.76,     # ✓ aus Notebook 40
        'rmse': 110.34,    # ✓ aus Notebook 40
        'r2':   0.3962,    # ✓ aus Notebook 40
        'color': COLORS[2],
        'notebook': 'NB40 – Jasmin'
    },
}

# Als DataFrame für Tabellen-Ausgabe
df_metrics = pd.DataFrame([
    {
        'Modell':    name.replace('\n', ' '),
        'Notebook':  v['notebook'],
        'MAE (€)':   v['mae'],
        'RMSE (€)':  v['rmse'],
        'R²':        v['r2'],
    }
    for name, v in METRICS.items()
])

print('Metriken-Übersicht:')
print(df_metrics.to_string(index=False))
df_metrics.to_csv(f'{CACHE_DIR}50_metrics_all_models.csv', index=False)
print(f'\nGespeichert → {CACHE_DIR}50_metrics_all_models.csv')

## 2 – Modellvergleich: Visualisierung

In [ ]:
model_labels = [k.replace('\n', '\n') for k in METRICS.keys()]
mae_vals  = [v['mae']  for v in METRICS.values()]
rmse_vals = [v['rmse'] for v in METRICS.values()]
r2_vals   = [v['r2']   for v in METRICS.values()]
bar_colors = [v['color'] for v in METRICS.values()]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Modellvergleich – MAE, RMSE und R² aller drei Modelle', fontsize=14, fontweight='bold', y=1.02)

# ── MAE ─────────────────────────────────────────────────────────────────────
bars0 = axes[0].bar(model_labels, mae_vals, color=bar_colors, edgecolor='white', width=0.5)
axes[0].set_title('MAE (Mean Absolute Error)')
axes[0].set_ylabel('Fehler (€)')
axes[0].bar_label(bars0, fmt='%.2f €', padding=4, fontsize=10)
axes[0].set_ylim(0, max(mae_vals) * 1.25)

# ── RMSE ────────────────────────────────────────────────────────────────────
bars1 = axes[1].bar(model_labels, rmse_vals, color=bar_colors, edgecolor='white', width=0.5)
axes[1].set_title('RMSE (Root Mean Squared Error)')
axes[1].set_ylabel('Fehler (€)')
axes[1].bar_label(bars1, fmt='%.2f €', padding=4, fontsize=10)
axes[1].set_ylim(0, max(rmse_vals) * 1.25)

# ── R² ──────────────────────────────────────────────────────────────────────
bars2 = axes[2].bar(model_labels, r2_vals, color=bar_colors, edgecolor='white', width=0.5)
axes[2].set_title('R² (Bestimmtheitsmaß)')
axes[2].set_ylabel('R²')
axes[2].bar_label(bars2, fmt='%.4f', padding=4, fontsize=10)
axes[2].set_ylim(0, min(max(r2_vals) * 1.3, 1.0))
axes[2].axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='R²=0.5 Referenz')
axes[2].legend(fontsize=9)

for ax in axes:
    ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}50_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot gespeichert ✓')

## 3 – Feature Importance (Modell III)

In [ ]:
feat_imp_df = pd.read_csv(f'{CACHE_DIR}40_feature_importance.csv')
top15 = feat_imp_df.head(15).copy()

# Lesbarere Feature-Namen
name_map = {
    'dist_city_center':         'Distanz Stadtzentrum',
    'dist_beach':               'Distanz Strand',
    'availability_365':         'Verfügbarkeit (365 Tage)',
    'review_activity_score':    'Review-Aktivitäts-Score',
    'calculated_host_listings_count': 'Anzahl Host-Listings',
    'number_of_reviews':        'Anzahl Bewertungen',
    'number_of_reviews_ltm':    'Bewertungen (letztes Jahr)',
    'reviews_per_month':        'Bewertungen pro Monat',
    'minimum_nights':           'Mindest-Nächte (roh)',
    'min_nights_capped':        'Mindest-Nächte (gekappt)',
    'latitude':                 'Breitengrad',
    'longitude':                'Längengrad',
    'is_licensed':              'Lizenziert (0/1)',
    'host_is_pro':              'Professioneller Host (0/1)',
    'room_type_Entire home/apt':'Zimmertyp: Ganze Wohnung',
    'room_type_Private room':   'Zimmertyp: Privatzimmer',
    'room_type_Hotel room':     'Zimmertyp: Hotelzimmer',
    'room_type_Shared room':    'Zimmertyp: Gemeinschaftszimmer',
}
top15['feature_label'] = top15['feature'].map(name_map).fillna(top15['feature'])

# Prozentuale Importance
top15['importance_pct'] = top15['importance'] / feat_imp_df['importance'].sum() * 100

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(
    top15['feature_label'][::-1],
    top15['importance_pct'][::-1],
    color='steelblue', edgecolor='white'
)
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('Relative Feature Importance (%)', fontsize=11)
ax.set_title('Top-15 Feature Importances – GradientBoostingRegressor\n(Modell III, Anteil an Gesamtvarianzreduktion)', fontsize=12, fontweight='bold')
ax.set_xlim(0, top15['importance_pct'].max() * 1.15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}50_feature_importance_clean.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot gespeichert ✓')

## 4 – Residuenanalyse

In [ ]:
# Modell & Daten neu laden für Residuenanalyse
import sys
sys.path.insert(0, '.')  # damit build_features importierbar wäre – hier inline

with open(f'{CACHE_DIR}40_model_III.pkl', 'rb') as f:
    model_III = pickle.load(f)

feature_names = pd.read_csv(f'{CACHE_DIR}40_feature_names.csv')['feature'].tolist()
print(f'Modell III geladen ✓ | {len(feature_names)} Features')

In [ ]:
# ── Daten für Residuen neu aufbauen ─────────────────────────────────────────
# (gleiche Pipeline wie in Notebook 40)
DATA_PATH = 'cache/10_listings_spanien_cleaned.csv'  # ← Pfad ggf. anpassen

from sklearn.model_selection import train_test_split

df_raw = pd.read_csv(DATA_PATH)

# Minimal-Preprocessing (identisch mit NB40)
CITY_CENTERS = {
    'Barcelona': (41.3851, 2.1734), 'Madrid': (40.4168, -3.7038),
    'Mallorca':  (39.5696, 2.6502), 'Menorca': (39.9496, 4.1156),
    'Girona':    (41.9794, 2.8214), 'Malaga':  (36.7213, -4.4214),
    'Sevilla':   (37.3891, -5.9845),'Valencia': (39.4699, -0.3763),
    'Euskadi':   (43.2630, -2.9350),
}
BEACH_HOTSPOTS = [
    (39.5296, 2.3873),(41.2754, 1.9862),(36.5007, -4.6679),
    (39.8628, 4.2591),(41.9787, 3.2169),(39.3560, -0.3319),
]
def haversine_approx(lat1, lon1, lat2, lon2):
    return np.sqrt((lat1-lat2)**2 + (lon1-lon2)**2)

df = df_raw.copy()
df['dist_city_center'] = df.apply(lambda r: haversine_approx(r['latitude'],r['longitude'],
    *CITY_CENTERS.get(r['region'],(r['latitude'],r['longitude']))), axis=1)
df['dist_beach'] = df.apply(lambda r: min(
    haversine_approx(r['latitude'],r['longitude'],b[0],b[1]) for b in BEACH_HOTSPOTS), axis=1)
df['is_licensed']           = df['license'].notna().astype(int)
df['host_is_pro']           = (df['calculated_host_listings_count'] > 1).astype(int)
df['min_nights_capped']     = df['minimum_nights'].clip(upper=30)
df['review_activity_score'] = df['reviews_per_month'].fillna(0) * df['availability_365']
df['reviews_per_month']     = df['reviews_per_month'].fillna(0)
df = pd.get_dummies(df, columns=['room_type','region'], drop_first=False)

X = df[feature_names].copy()
y = df['price'].copy()
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_pred = model_III.predict(X_test)
residuals = y_test - y_pred
print(f'Testset neu aufgebaut: {X_test.shape[0]:,} Samples ✓')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Residuenanalyse – GradientBoostingRegressor (Modell III)', fontsize=13, fontweight='bold')

# ── Residuen vs. Predicted ──────────────────────────────────────────────────
axes[0].scatter(y_pred, residuals, alpha=0.1, s=8, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Vorhergesagter Preis (€)')
axes[0].set_ylabel('Residuum (€)')
axes[0].set_title('Residuen vs. Predicted')

# ── Residuen-Histogramm ─────────────────────────────────────────────────────
axes[1].hist(residuals, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].axvline(residuals.mean(), color='orange', linestyle='-', linewidth=1.5,
                label=f'Mittelwert: {residuals.mean():.1f} €')
axes[1].set_xlabel('Residuum (€)')
axes[1].set_ylabel('Häufigkeit')
axes[1].set_title('Verteilung der Residuen')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}50_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot gespeichert ✓')

## 5 – Preisvorhersage nach Region & Zimmertyp
*(Business Insights: Welche Kombination ist am lukrativsten?)*

In [ ]:
# Tatsächliche Medianpreise je Region & Zimmertyp aus dem Testset
df_test = X_test.copy()
df_test['price_actual']    = y_test.values
df_test['price_predicted'] = y_pred

# Region & Zimmertyp zurückgewinnen
region_cols   = [c for c in df_test.columns if c.startswith('region_')]
roomtype_cols = [c for c in df_test.columns if c.startswith('room_type_')]

df_test['region']    = df_test[region_cols].idxmax(axis=1).str.replace('region_', '')
df_test['room_type'] = df_test[roomtype_cols].idxmax(axis=1).str.replace('room_type_', '')

# Median-Preise je Region
region_price = df_test.groupby('region')[['price_actual','price_predicted']].median().sort_values('price_actual', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(region_price))
w = 0.35
ax.bar(x - w/2, region_price['price_actual'],    width=w, label='Tatsächlich (Median)', color='steelblue')
ax.bar(x + w/2, region_price['price_predicted'], width=w, label='Vorhergesagt (Median)', color='#D65F5F', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(region_price.index, rotation=20, ha='right')
ax.set_ylabel('Preis (€)')
ax.set_title('Medianer Tatsächlicher vs. Vorhergesagter Preis je Region', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}50_price_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMedianpreise je Region:')
print(region_price.round(2).to_string())

In [ ]:
# Median-Preise je Zimmertyp
roomtype_price = df_test.groupby('room_type')[['price_actual','price_predicted']].median().sort_values('price_actual', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(roomtype_price))
bars_a = ax.bar(x - 0.2, roomtype_price['price_actual'],    width=0.35, label='Tatsächlich', color='steelblue')
bars_p = ax.bar(x + 0.2, roomtype_price['price_predicted'], width=0.35, label='Vorhergesagt', color='#D65F5F', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(roomtype_price.index, rotation=10)
ax.set_ylabel('Preis (€)')
ax.set_title('Medianer Preis je Zimmertyp', fontweight='bold')
ax.legend()
ax.bar_label(bars_a, fmt='%.0f€', padding=3, fontsize=9)
ax.bar_label(bars_p, fmt='%.0f€', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}50_price_by_roomtype.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMedianpreise je Zimmertyp:')
print(roomtype_price.round(2).to_string())

## 6 – Zusammenfassung aller Outputs

In [ ]:
print('=' * 60)
print('  NOTEBOOK 50 – ALLE OUTPUTS')
print('=' * 60)
print()
print('  PLOTS (outputs/):')
print('    50_model_comparison.png         → Kapitel 3.1')
print('    50_feature_importance_clean.png → Kapitel 3.2')
print('    50_residuals.png                → Kapitel 3.1')
print('    50_price_by_region.png          → Kapitel 4')
print('    50_price_by_roomtype.png        → Kapitel 4')
print()
print('  DATEN (cache/):')
print('    50_metrics_all_models.csv       → LaTeX-Tabelle Kap. 3.1')
print()
print('  METRIKEN-ÜBERSICHT:')
print()
print(df_metrics.to_string(index=False))
print()
print('  TODO:')
print('    [ ] Werte von Annika (Modell I)  in Zelle 1 eintragen')
print('    [ ] Werte von Mareike (Modell II) in Zelle 1 eintragen')
print('    [ ] Dann: Run All → alle Plots aktualisieren sich automatisch')
print('=' * 60)